> 在线讲义：[第 3 章 · 运控里的 PPO 配方：高斯策略、几千个环境、按 KL 调学习率](https://wikieden.github.io/robot-motion-rl/lessons/03-ppo-recipe.html) · 本书《机器人运控与强化学习》

# 第 3 章 · 运控里的 PPO 配方：高斯策略、几千个环境、按 KL 调学习率

配套讲义：仓库里 `robot-motion-rl/notes/lessons/03-ppo-recipe.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

（μ、σ、a 是编的，批量和 KL 规则取自 legged_gym）

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 log 概率 | 三项之和 | `gauss_logp(0.7, 0.2, 0.5)` | −0.726 |
| 第二步 比率 | exp(log 差) | `ratio(0.7, 0.3, 0.2, 0.5)` | 1.197；μ = 0.45 时 1.455，剪成 1.200 |
| 第三步 批量账 | 4096 × 24 | `batch_accounting()` | (98304, 24576, 20) |
| 第四步 调学习率 | Algorithm 1 | `adaptive_lr(1e-3, kl)` | 6.67e-4、1e-3、1.5e-3 |

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `gauss_logp`**：一维高斯 log 概率：−(a−μ)²/(2σ²) − log σ − ½ log(2π)；多维时各维相加。

In [ ]:
def gauss_logp(a, mu, sigma):
    """一维高斯 log 概率：−(a−μ)²/(2σ²) − log σ − ½ log(2π)；多维时各维相加。"""
    a, mu, sigma = np.asarray(a, float), np.asarray(mu, float), np.asarray(sigma, float)
    return float(np.sum(-((a - mu) ** 2) / (2 * sigma**2) - np.log(sigma) - 0.5 * np.log(2 * np.pi)))

**函数 `ratio`**：比率 r = π_新(a)/π_旧(a) = exp(log π_新 − log π_旧)。

In [ ]:
def ratio(a, mu_new, mu_old, sigma):
    """比率 r = π_新(a)/π_旧(a) = exp(log π_新 − log π_旧)。"""
    return float(np.exp(gauss_logp(a, mu_new, sigma) - gauss_logp(a, mu_old, sigma)))

**函数 `ppo_term`**：PPO 剪裁目标的一项：min(r·A, clip(r, 1−ε, 1+ε)·A)。

In [ ]:
def ppo_term(r, adv, eps=0.2):
    """PPO 剪裁目标的一项：min(r·A, clip(r, 1−ε, 1+ε)·A)。"""
    return min(r * adv, float(np.clip(r, 1 - eps, 1 + eps)) * adv)

**函数 `batch_accounting`**：一次迭代的批量账：总样本、每个 mini-batch 大小、梯度步数。

In [ ]:
def batch_accounting(envs=4096, steps=24, minibatches=4, epochs=5):
    """一次迭代的批量账：总样本、每个 mini-batch 大小、梯度步数。"""
    total = envs * steps
    return total, total // minibatches, minibatches * epochs

**函数 `adaptive_lr`**：按 KL 调学习率（rsl_rl 的规则）：KL > 2·目标就除以 1.5，KL < 目标/2 就乘 1.5。

In [ ]:
def adaptive_lr(lr, kl, kl_target=0.01, factor=1.5):
    """按 KL 调学习率（rsl_rl 的规则）：KL > 2·目标就除以 1.5，KL < 目标/2 就乘 1.5。"""
    if kl > 2 * kl_target:
        return max(lr / factor, 1e-5)
    if kl < kl_target / 2:
        return min(lr * factor, 1e-2)
    return lr

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 高斯策略的 log 概率")
print(f"μ=0.2, σ=0.5, a=0.7 → log π = {gauss_logp(0.7, 0.2, 0.5):.3f}")

In [ ]:
print("\n第二步 比率与剪裁")
r = ratio(0.7, 0.3, 0.2, 0.5)
print(f"μ 从 0.2 改到 0.3 → r = {r:.3f}；A=+1 时 PPO 项 = {ppo_term(r, 1.0):.3f}")
r2 = ratio(0.7, 0.45, 0.2, 0.5)
print(f"μ 改到 0.45 → r = {r2:.3f}；A=+1 时被剪成 {ppo_term(r2, 1.0):.3f}")

In [ ]:
print("\n第三步 批量账")
total, mb, steps = batch_accounting()
print(f"4096 × 24 = {total} 个样本；mini-batch {mb}；每次迭代 {steps} 步梯度")

In [ ]:
print("\n第四步 按 KL 调学习率")
for kl in (0.03, 0.01, 0.004):
    print(f"lr=1e-3, KL={kl} → {adaptive_lr(1e-3, kl):.2e}")

## 3. 代码里才看得到的

- σ 不变时，log 概率的后两项（−log σ、常数）在新旧策略里相同，相减后消掉，比率只取决于平方项。
- 学习率有上下限（1e-5、1e-2），否则 KL 一直偏小时学习率会无限变大。

## 4. 自己改一改

- 让新策略的 σ 也变（比如 0.5 → 0.4），看同一个 a 的比率怎么变。
- 用 `adaptive_lr` 模拟 KL 一直是 0.001 的十次更新，看 lr 多久撞到上限。

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 连续动作投飞镖：瞄准 μ、手抖 σ，log 概率三项加；
> 比率 log 相减再 exp，过了一点二就剪平；
> 四千个环境二十四步，切四份看五遍；
> KL 超两倍降一点五，低一半升一点五。